# 03. factor performance の計算

`data/raw/` から NAM・common ファクターのリターンと IC・TC を計算し、`data/processed/<output_ymd>/factor/` に保存します。
パラメータを変えて、すべて実行してください。図で強調する期間は 04 で決めます。

| 指標 | 内容 |
|---|---|
| NAM | Region ごとの Q5 (RRI はクラス 1, 2) の等ウェイトリターンから、ユニバースの等ウェイト平均を引いた超過リターン (トータル / スペシフィック) |
| common | ファクターごとの単回帰の傾き (スコア 1σ あたり) と、ファクターグループの単純平均 |
| IC | 各日に使うスコアと当日の USD リターンの相関 (Pearson) |
| TC | アクティブウェイト (ファンド − BM) とスコアの相関。アクティブウェイトが 0 の銘柄は除く |
| リバランス相関 | `reb_ymd` 以降で最初の日の日次スコアと、各日の日次スコアの相関 |

In [ ]:
output_ymd = 20261125  # 保存先の日付 (data/processed/<日付>/factor/)
from_ymd = 20260101  # 計算開始日。図の推移もこの日から描く (IC の 20 日移動平均のため長めに取る)
to_ymd = 20261124  # 計算終了日
score_freq = "monthly"  # NAM のスコア: "monthly" (直近の月末) / "daily" (当日)。IC・TC にも使う
reb_ymd = None  # リバランス日 (YYYYMMDD)。None ならリバランス相関を計算しない

In [ ]:
# em_market のコードを変更したとき、カーネルを再起動せずに反映する
%load_ext autoreload
%autoreload 2

import pandas as pd

from em_market.paths import processed_dir
from em_market.reports.factor import build_factor_results
from em_market.viz.factor_monitor import NAM_LABELS

pd.set_option("display.float_format", "{:.4f}".format)

## 計算と保存

In [ ]:
results = build_factor_results(
    from_ymd, to_ymd, score_freq=score_freq, reb_ymd=reb_ymd
)
output_dir = processed_dir(pd.Timestamp(str(output_ymd)), "factor")
paths = results.save(output_dir)
print(f"Saved {len(paths)} files to {output_dir}")

## 確認

直近 5 営業日の日次リターンと、計算期間の累積 (日次リターン % の単純和)。

In [ ]:
results.nam_total.tail().rename(columns=NAM_LABELS)

In [ ]:
pd.DataFrame(
    {
        "total": results.nam_total.sum(min_count=1),
        "specific": results.nam_specific.sum(min_count=1),
    }
).rename(index=NAM_LABELS)

In [ ]:
results.common_groups.sum(min_count=1).sort_values(ascending=False).to_frame("cumulative")

In [ ]:
# IC・TC・リバランス相関
summary = {
    "IC mean": results.ic.mean(),
    "IC mean (last 20 days)": results.ic.tail(20).mean(),
    "TC (latest)": results.tc.iloc[-1],
}
if results.rebalance is not None:
    summary["rebalance corr (latest)"] = results.rebalance.iloc[-1]
pd.DataFrame(summary).rename(index=NAM_LABELS)